# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content file (an article belonging to a client) describing its features such as keyword,URL and content attribute at a point time , it is not one row per day or client or query.

Note:Each client has a one-to-many relationship with each content file i.e each client could have more than one article.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/dim_content.csv')
df.columns


Index(['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id',
       'keyword_char_count', 'keyword_token_count', 'url_char_count',
       'content_created_date', 'content_updated_date', 'content_type',
       'search_volume', 'competition', 'competition_level', 'cpc',
       'main_intent', 'backlinks', 'category_count', 'keyword_created_date',
       'provider_used', 'model_used', 'char_count', 'word_count',
       'last_optimized_date', 'optimization_eligible_date', 'is_published',
       'is_deleted'],
      dtype='object')

In [3]:
if df['content_hash_id'].nunique() == len(df):
  print(f'''One row stands for one content file(i.e one article belonging to a client),
   because the Number of unique content id({df['content_hash_id'].nunique()}) is the same as the number of rows({len(df)})''')

else:
  print('False claim')

One row stands for one content file(i.e one article belonging to a client),
   because the Number of unique content id(519606) is the same as the number of rows(519606)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature:
1. search_volume
2. competition
3. cpc
4. word_count
5. content_created_date

Label:
  Since I am working with an Unsupervised Learning Algorithm(Archetype Clustering), hence there will be no label but ;
  Proxy (for sucess ) are:
      clusters are stable across re-runs and resampling and each one is interpretable as a distinct content archetype.

Context:
  The content used for training will be content that falls within the range content_created_date < 01/07/2026.

Excluded:
1.char_count: To prevent redundancy because char_count more or less information as word_count.
2.keyword_char_count, keyword_token_count, url_char_count: these are features explaining what could potentially affect the search volume, which already has a column of it own.
3.category_count, backlinks:I can't say for sure what each of them stands for, hence I excluded them to prevent ambiguity.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Feature Frame:
search_volume,competition, cpc: these are knowable at decision moment because keyword research has happened before the content existed, so they are already available before decision moment.
word_contract:this is also knowable at decision moment because the the article are already written before clustering.
content_created_date: this also knowable at decision moment, because the creation date is fixed the moment the content is created - it can't be change afterwards.  

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['content_created_date'] = pd.to_datetime(df['content_created_date'])
windowed = (df[df['content_created_date'] < '2026-07-01']).copy()

total_rows = len(windowed)
distinct_ids = windowed['content_hash_id'].nunique()

print(total_rows, distinct_ids)

515751 515751


In [6]:
import duckdb

query = '''
SELECT
  COUNT (*)                         AS total_rows,
  COUNT(DISTINCT content_hash_id) AS distinct_ids
FROM df
WHERE content_created_date < '2026-07-01';
'''

result = duckdb.query(query).df()
print(result)

   total_rows  distinct_ids
0      515751        515751


In [7]:
query_2 = '''
SELECT
  COUNT (*)                         AS row_count,
  MIN(content_created_date) AS earliest_date,
  MAX(content_created_date) AS latest_date
FROM df
WHERE content_created_date < '2026-07-01';
'''

result_2 = duckdb.query(query_2).df()
print(result_2)

   row_count earliest_date latest_date
0     515751    2024-10-16  2026-06-30


In [8]:
query_3 = '''
SELECT
  COUNT (*) AS total_rows,
  COUNT (*) FILTER (WHERE is_published IS TRUE) AS available_rows
FROM df
WHERE content_created_date < '2026-07-01';
'''

result_3 = duckdb.query(query_3).df()
print(result_3)

   total_rows  available_rows
0      515751          407685


In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import robust_scale, RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import numpy as np

In [10]:
date = pd.to_datetime('2026-07-01')
windowed['content_created_date'] = (date - pd.to_datetime(windowed['content_created_date'])).dt.days
features = ['search_volume', 'competition', 'cpc', 'word_count', 'content_created_date']
X = windowed[features].copy().replace([np.inf, -np.inf], np.nan).fillna(0)
X_log = np.log1p(X+1)
scaler = RobustScaler()
x_scaled = scaler.fit_transform(X_log)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster = kmeans.fit_predict(x_scaled)

In [11]:
id_numeric = windowed['content_hash_id'].astype('category').cat.codes.values.reshape(-1, 1)
x_leak = np.column_stack([x_scaled, id_numeric*10])
cluster_leak = kmeans.fit_predict(x_leak)

In [12]:
print(f'normal={silhouette_score(x_scaled, cluster)}, leaked = {silhouette_score(x_leak, cluster_leak)}')

KeyboardInterrupt: 

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The slice has no performance or engagement data, because the slice is based on how content was made not based on how it performs, hence two content may fall in the same archetype and behave in a different way after publishing , since that fact lies in a separate fact table that is beyond the scope of this table.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.